# 06 — Supervised Balanced Random Forest

This notebook:
- loads the prepared train, validation, and test splits,
- trains the binary Balanced Random Forest model,
- evaluates validation and test performance,
- plots feature importance,
- compares SBS96 importance with the COSMIC SBS4 profile.


In [ ]:
%matplotlib inline

from pathlib import Path
import re

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import Image, display, Markdown

PROJECT_ROOT = Path.cwd().resolve()

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

def show_image(path, width=1000):
    path = Path(path)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"Image not found: {path}")

def show_images(*paths, width=1000):
    for path in paths:
        show_image(path, width=width)

def get_sbs_columns(columns):
    cols = [col for col in columns if str(col).startswith("SBS") and str(col)[3:].isdigit()]
    return sorted(cols, key=lambda col: int(str(col)[3:]))

def normalize_rows(frame):
    frame = frame.copy()
    frame = frame.apply(pd.to_numeric, errors="coerce").fillna(0.0)
    row_sums = frame.sum(axis=1)
    frame = frame.loc[row_sums > 0].copy()
    return frame.div(frame.sum(axis=1), axis=0)

def extract_patient_id(value):
    match = re.search(r"(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})", str(value))
    return match.group(1) if match else np.nan

def map_smoking_label(value):
    if pd.isna(value):
        return "Unknown"

    text = str(value).strip().lower()

    if text in {"", "nan", "not reported", "unknown"}:
        return "Unknown"
    if "never" in text or "lifelong" in text or "non-smoker" in text or "nonsmoker" in text:
        return "Never"
    if "current reformed" in text or "reformed" in text or "former" in text:
        return "Former"
    if "current" in text:
        return "Current"
    return "Unknown"

def set_spaced_xticks(ax, labels, step=3):
    ax.set_xticks(np.arange(len(labels)) + 0.5)
    ax.set_xticklabels(labels, rotation=90)
    for i, label in enumerate(ax.get_xticklabels()):
        if i % step != 0:
            label.set_visible(False)

    from scipy.stats import spearmanr
    from sklearn.inspection import permutation_importance
    from sklearn.metrics import (
        accuracy_score,
        balanced_accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        roc_curve,
        auc,
    )
    from sklearn.metrics.pairwise import cosine_similarity
    from imblearn.ensemble import BalancedRandomForestClassifier


## 1. Define paths


In [ ]:
split_output_dir = PROJECT_ROOT / "results" / "brf_split_binary"
plot_dir = PROJECT_ROOT / "plots" / "brf_split_binary"
cosmic_sbs4_path = PROJECT_ROOT / "data" / "v3.2_SBS4_DIFFERENCE.txt"

plot_dir.mkdir(parents=True, exist_ok=True)

train_path = split_output_dir / "train.csv"
val_path = split_output_dir / "val.csv"
test_path = split_output_dir / "test.csv"

print("Train file :", train_path)
print("Val file   :", val_path)
print("Test file  :", test_path)
print("Plot dir   :", plot_dir)
print("COSMIC SBS4:", cosmic_sbs4_path)


## 2. Load the split tables


In [ ]:
train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

sbs_cols = get_sbs_columns(train_df.columns)

print("Train shape:", train_df.shape)
print("Val shape  :", val_df.shape)
print("Test shape :", test_df.shape)
print("Number of SBS columns:", len(sbs_cols))


## 3. Prepare the feature matrices


In [ ]:
def prep_features(frame, sbs_cols, age_median=None, gender_fill_value=None):
    frame = frame.copy()

    frame["demographic.age_at_index"] = pd.to_numeric(
        frame["demographic.age_at_index"],
        errors="coerce",
    )

    frame["demographic.gender"] = (
        frame["demographic.gender"]
        .astype(str)
        .str.lower()
        .map({"male": 1, "female": 0})
    )

    if age_median is None:
        age_median = frame["demographic.age_at_index"].median()

    if gender_fill_value is None:
        mode_values = frame["demographic.gender"].mode()
        gender_fill_value = mode_values.iloc[0] if not mode_values.empty else 0

    frame["demographic.age_at_index"] = frame["demographic.age_at_index"].fillna(age_median)
    frame["demographic.gender"] = frame["demographic.gender"].fillna(gender_fill_value)

    x_sbs = frame[sbs_cols].copy()
    x_sbs = x_sbs.div(x_sbs.sum(axis=1), axis=0).fillna(0.0)

    x_clin = pd.DataFrame(
        {
            "age_years": frame["demographic.age_at_index"],
            "demographic.gender": frame["demographic.gender"],
        },
        index=frame.index,
    )

    x = pd.concat([x_sbs, x_clin], axis=1)
    y = frame["Smoking_Bin"].astype(int).to_numpy()

    return x, y, age_median, gender_fill_value


In [ ]:
X_train, y_train, age_median, gender_fill_value = prep_features(train_df, sbs_cols)
X_val, y_val, _, _ = prep_features(val_df, sbs_cols, age_median, gender_fill_value)
X_test, y_test, _, _ = prep_features(test_df, sbs_cols, age_median, gender_fill_value)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)


## 4. Run the manual grid search on the validation set


In [ ]:
RANDOM_STATE = 42

results = []
best_score = -1.0
best_params = None
best_model = None

for n_estimators in [200, 400, 800]:
    for max_depth in [None, 20]:
        params = {
            "n_estimators": n_estimators,
            "max_features": "sqrt",
            "max_depth": max_depth,
        }

        model = BalancedRandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1,
            sampling_strategy="auto",
            replacement=False,
            bootstrap=True,
            **params,
        )

        model.fit(X_train, y_train)

        val_pred = model.predict(X_val)
        val_proba = model.predict_proba(X_val)[:, 1]

        score_f1 = f1_score(y_val, val_pred, average="macro")
        score_bal = balanced_accuracy_score(y_val, val_pred)

        fpr, tpr, _ = roc_curve(y_val, val_proba)
        score_auc = auc(fpr, tpr)

        row = params.copy()
        row["val_f1_macro"] = score_f1
        row["val_balanced_accuracy"] = score_bal
        row["val_auc"] = score_auc
        results.append(row)

        if score_f1 > best_score:
            best_score = score_f1
            best_params = params
            best_model = model

grid_df = pd.DataFrame(results).sort_values("val_f1_macro", ascending=False).reset_index(drop=True)
display(grid_df)

print("Best parameters:", best_params)
print("Best validation macro F1:", round(best_score, 4))


## 5. Define evaluation helpers


In [ ]:
def save_report_and_cm_binary(y_true, y_pred, tag):
    accuracy = accuracy_score(y_true, y_pred)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)

    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["Never", "Ever"],
        zero_division=0,
    )

    report_path = split_output_dir / f"{tag}_report.txt"
    with open(report_path, "w", encoding="utf-8") as handle:
        handle.write(f"accuracy: {accuracy:.4f}\n")
        handle.write(f"balanced_accuracy: {balanced_acc:.4f}\n\n")
        handle.write(report)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Never", "Ever"],
        yticklabels=["Never", "Ever"],
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"{tag} confusion matrix")
    plt.tight_layout()

    cm_path = plot_dir / f"{tag}_confusion_matrix.png"
    plt.savefig(cm_path, dpi=300, bbox_inches="tight")
    plt.show()

    return {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_acc,
        "report_path": report_path,
        "cm_path": cm_path,
    }

def plot_binary_roc(y_true, y_score, tag):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(5, 4))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{tag} ROC curve")
    plt.legend(loc="lower right")
    plt.tight_layout()

    roc_path = plot_dir / f"{tag}_roc.png"
    plt.savefig(roc_path, dpi=300, bbox_inches="tight")
    plt.show()

    auc_path = split_output_dir / f"{tag}_auc.tsv"
    pd.DataFrame([{"auc": roc_auc}]).to_csv(auc_path, sep="\t", index=False)

    return {"auc": roc_auc, "roc_path": roc_path, "auc_path": auc_path}


## 6. Training-set sanity check


In [ ]:
brf_selected = best_model

train_pred = brf_selected.predict(X_train)
train_metrics = save_report_and_cm_binary(y_train, train_pred, tag="TRAIN_FIT_SANITY")

display(pd.DataFrame([train_metrics]))


## 7. Validation-set evaluation


In [ ]:
val_pred = brf_selected.predict(X_val)
val_proba = brf_selected.predict_proba(X_val)[:, 1]

val_metrics = save_report_and_cm_binary(y_val, val_pred, tag="VAL")
val_roc = plot_binary_roc(y_val, val_proba, tag="VAL")

display(pd.DataFrame([{**val_metrics, **val_roc}]))
display(pd.read_csv(split_output_dir / "VAL_auc.tsv", sep="\t"))


## 8. Train the final model on train + validation and evaluate the test set


In [ ]:
X_trainval = pd.concat([X_train, X_val], axis=0)
y_trainval = np.concatenate([y_train, y_val], axis=0)

brf_final = BalancedRandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    sampling_strategy="auto",
    replacement=False,
    bootstrap=True,
    **best_params,
)

brf_final.fit(X_trainval, y_trainval)

test_pred = brf_final.predict(X_test)
test_proba = brf_final.predict_proba(X_test)[:, 1]

test_metrics = save_report_and_cm_binary(y_test, test_pred, tag="TEST_FINAL")
test_roc = plot_binary_roc(y_test, test_proba, tag="TEST_FINAL")

display(pd.DataFrame([{**test_metrics, **test_roc}]))
display(pd.read_csv(split_output_dir / "TEST_FINAL_auc.tsv", sep="\t"))


## 9. Plot impurity-based feature importance


In [ ]:
fi = pd.Series(brf_final.feature_importances_, index=X_train.columns).sort_values(ascending=False)
fi.to_csv(split_output_dir / "feature_importance_final_model.tsv", sep="\t", header=["importance"])

top30 = fi.head(30)

plt.figure(figsize=(10, 9))
sns.barplot(x=top30.values, y=top30.index)
plt.title("Top 30 feature importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()

fi_plot_path = plot_dir / "feature_importance_top30_final.png"
plt.savefig(fi_plot_path, dpi=300, bbox_inches="tight")
plt.show()

display(fi.head(15).rename("importance").reset_index().rename(columns={"index": "feature"}))


## 10. Plot permutation importance


In [ ]:
perm = permutation_importance(
    brf_final,
    X_test,
    y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

perm_importance = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
perm_importance.to_csv(split_output_dir / "feature_importance_permutation.tsv", sep="\t", header=["importance"])

top30_perm = perm_importance.head(30)

plt.figure(figsize=(10, 9))
sns.barplot(x=top30_perm.values, y=top30_perm.index)
plt.title("Top 30 permutation importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()

perm_plot_path = plot_dir / "feature_importance_top30_permutation.png"
plt.savefig(perm_plot_path, dpi=300, bbox_inches="tight")
plt.show()

display(
    perm_importance.head(15)
    .rename("importance")
    .reset_index()
    .rename(columns={"index": "feature"})
)


## 11. Focus on SBS96 importance only


In [ ]:
fi_sbs = fi.reindex(sbs_cols).dropna().sort_values(ascending=False)
fi_sbs.to_csv(split_output_dir / "feature_importance_sbs96_only.tsv", sep="\t", header=["importance"])

top20_sbs = fi_sbs.head(20)

plt.figure(figsize=(10, 8))
sns.barplot(x=top20_sbs.values, y=top20_sbs.index)
plt.title("Top 20 SBS96 feature importances")
plt.xlabel("Importance")
plt.ylabel("SBS96 channel")
plt.tight_layout()

sbs_plot_path = plot_dir / "feature_importance_sbs96_top20.png"
plt.savefig(sbs_plot_path, dpi=300, bbox_inches="tight")
plt.show()

display(
    fi_sbs.head(20)
    .rename("importance")
    .reset_index()
    .rename(columns={"index": "SBS96_channel"})
)


## 12. Compare the SBS96 importance profile with COSMIC SBS4


In [ ]:
cosmic = pd.read_csv(cosmic_sbs4_path, sep="\t", index_col=0)

if "Signature_4_GRCh38" not in cosmic.columns:
    raise KeyError("Column 'Signature_4_GRCh38' is missing in the COSMIC SBS4 file.")

cosmic_sbs4 = cosmic["Signature_4_GRCh38"].copy()
cosmic_sbs4.index = cosmic_sbs4.index.astype(str)
cosmic_sbs4 = cosmic_sbs4.reindex(sbs_cols)

if cosmic_sbs4.isna().any():
    missing_channels = cosmic_sbs4[cosmic_sbs4.isna()].index.tolist()
    raise RuntimeError(f"Missing SBS4 values for channels: {missing_channels}")

imp_vec = fi_sbs.reindex(sbs_cols).astype(float)
imp_vec = imp_vec / imp_vec.sum()

sbs4_vec = pd.to_numeric(cosmic_sbs4, errors="coerce").astype(float)
sbs4_vec = sbs4_vec / sbs4_vec.sum()

TOP_N_OVERLAP = 20
top_imp = set(imp_vec.sort_values(ascending=False).head(TOP_N_OVERLAP).index)
top_sbs4 = set(sbs4_vec.sort_values(ascending=False).head(TOP_N_OVERLAP).index)
overlap = sorted(top_imp & top_sbs4)

spearman_r, spearman_p = spearmanr(imp_vec.values, sbs4_vec.values)
cos_sim = cosine_similarity(
    imp_vec.values.reshape(1, -1),
    sbs4_vec.values.reshape(1, -1),
)[0, 0]

metrics_df = pd.DataFrame(
    [
        {
            "top_n": TOP_N_OVERLAP,
            "n_overlap": len(overlap),
            "spearman_r": spearman_r,
            "spearman_p": spearman_p,
            "cosine_similarity": cos_sim,
        }
    ]
)

metrics_df.to_csv(split_output_dir / "sbs4_overlap_metrics.tsv", sep="\t", index=False)
pd.DataFrame({"overlap_channel": overlap}).to_csv(
    split_output_dir / f"sbs4_overlap_top{TOP_N_OVERLAP}_channels.tsv",
    sep="\t",
    index=False,
)

print("Overlap channels:")
display(pd.DataFrame({"overlap_channel": overlap}))

print("Summary metrics:")
display(metrics_df)
